# Galerie données mixtes — 02 · Le coût des erreurs n'est pas symétrique 🟢

> **Étagère optionnelle post-M4** — pas un brief, pas de livrable, pas de note.
> **Autonomie** : 🟢 **résolu** — tu lis, tu exécutes, tu comprends chaque cellule.
> **Durée** : ~1 h 30 · **Pré-requis** : avoir fait le notebook 01.
> **Fiches à garder ouvertes** : `fiche_desequilibre_classes.pdf` ·
> `cheatsheet_metriques.md` (compromis précision ↔ rappel) ·
> `fiche_validation_reglage.pdf` · canvas certif §7.2 (fallback)

## Le contexte

Suite de la démo du notebook 01. Marc Delval, directeur général de
**NovaThread**, coupe la présentation :

> « Votre f1-machin ne m'intéresse pas. Une cliente furieuse qu'on ne rappelle
> pas, c'est ~80 € de perdus (churn + bouche-à-oreille). Un rappel inutile,
> c'est 4 € de temps SAV. **Réglez votre modèle en conséquence.** »

Marc vient de te donner une **matrice de coûts**. Ce notebook montre les deux
leviers pour en tenir compte — **sans jamais toucher au test avant le verdict
final** :

1. **Poids de classes** (`class_weight`) : on ré-entraîne en pénalisant plus
   les erreurs sur la classe critique ;
2. **Seuil de décision** sur `predict_proba` : on ne ré-entraîne **rien**, on
   déplace le curseur de décision — réglable en production à tout moment.

## Setup + rechargement express

Chargement, cible, anti-fuite, split : tout est détaillé dans le notebook 01 —
ici on condense en deux cellules.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_STATE = 42

URL = ("https://raw.githubusercontent.com/AFAgarap/ecommerce-reviews-analysis/"
       "master/Womens%20Clothing%20E-Commerce%20Reviews.csv")

try:
    df = pd.read_csv(URL, index_col=0)
except Exception as err:
    print(f"Téléchargement impossible ({err}) — lecture du CSV local.")
    df = pd.read_csv("data/clothing_reviews.csv", index_col=0)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


def vers_satisfaction(note: int) -> str:
    if note <= 2:
        return "insatisfaite"
    if note == 3:
        return "mitigée"
    return "satisfaite"


df["satisfaction"] = df["rating"].apply(vers_satisfaction)
df = df.drop(columns=["rating", "recommended_ind", "positive_feedback_count"])
df["review_text"] = df["review_text"].fillna("")
df["longueur_avis"] = df["review_text"].str.len()
print(df.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

colonnes_num = ["age", "longueur_avis"]
colonnes_cat = ["division_name", "department_name"]
colonne_txt = "review_text"


def fabrique_preparation() -> ColumnTransformer:
    return ColumnTransformer([
        ("num", Pipeline([("imputation", SimpleImputer(strategy="median")),
                          ("echelle", StandardScaler())]), colonnes_num),
        ("cat", Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), colonnes_cat),
        ("txt", TfidfVectorizer(max_features=5000, min_df=3, stop_words="english"), colonne_txt),
    ])


X = df[colonnes_num + colonnes_cat + [colonne_txt]]
y = df["satisfaction"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# ⚠️ Les réglages (poids, seuils) se cherchent sur un jeu de VALIDATION,
# jamais sur le test — le test reste scellé jusqu'au verdict final.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)
print("réglage :", X_tr.shape, "| validation :", X_val.shape, "| test scellé :", X_test.shape)

## [1] La baseline et son angle mort

Régression logistique **sans aucun réglage** : c'est le point de comparaison.
Compte les insatisfaites **manquées** (prédites satisfaites) : chacune vaut
80 € pour Marc.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ordre = ["insatisfaite", "mitigée", "satisfaite"]

pipe_brut = Pipeline([("preparation", fabrique_preparation()),
                      ("modele", LogisticRegression(max_iter=2000))])
pipe_brut.fit(X_tr, y_tr)
pred_val = pipe_brut.predict(X_val)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_val, pred_val, labels=ordre,
                                        ax=ax, colorbar=False)
ax.set_title("Baseline sans réglage — validation")
plt.tight_layout()

manquees = ((y_val == "insatisfaite") & (pred_val == "satisfaite")).sum()
total_insat = (y_val == "insatisfaite").sum()
print(f"Insatisfaites jamais rappelées : {manquees}/{total_insat} "
      f"({manquees / total_insat:.0%}) → {manquees * 80} € perdus sur ce seul échantillon")

Sans réglage, le modèle optimise le taux de bonnes réponses global… donc il
sacrifie la classe rare. C'est exactement l'angle mort de l'accuracy vu dans
`fiche_desequilibre_classes.pdf`.

## [2] Chiffrer : la matrice de coûts métier

On traduit la phrase de Marc en euros, case par case (lignes = vérité,
colonnes = prédiction). Les valeurs exactes se **négocient avec le métier** —
l'important est l'**asymétrie**, pas la précision comptable.

In [ ]:
from sklearn.metrics import confusion_matrix

couts = pd.DataFrame(
    [[0, 10, 80],   # vérité : insatisfaite  (mitigée = rappel tardif, satisfaite = perdue)
     [4,  0,  8],   # vérité : mitigée
     [4,  2,  0]],  # vérité : satisfaite    (insatisfaite = rappel inutile à 4 €)
    index=pd.Index(ordre, name="vérité"),
    columns=pd.Index(ordre, name="prédiction"),
)
couts

In [ ]:
def cout_total(y_vrai, y_pred) -> int:
    """Coût métier total en € : matrice de confusion × matrice de coûts."""
    conf = confusion_matrix(y_vrai, y_pred, labels=ordre)
    return int((conf * couts.to_numpy()).sum())


print(f"Coût de la baseline sur la validation : {cout_total(y_val, pred_val)} €")

Une seule ligne de NumPy, et ton tableau de bord parle enfin la langue du DG.
Ce `cout_total` devient **la** métrique d'arbitrage des deux leviers.

## [3] Levier 1 — les poids de classes

On ré-entraîne en donnant un poids croissant à `insatisfaite`. Observe les
colonnes **en même temps** : le rappel monte, la précision s'effondre, le
`f1_macro` se dégrade… et le coût ? C'est lui qui tranche.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

lignes = []
for poids in [1, 3, 10, 30, 100]:
    pipe = Pipeline([
        ("preparation", fabrique_preparation()),
        ("modele", LogisticRegression(max_iter=2000,
                                      class_weight={"insatisfaite": poids,
                                                    "mitigée": 1,
                                                    "satisfaite": 1})),
    ])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_val)
    lignes.append({
        "poids insatisfaite": poids,
        "rappel insat.": round(recall_score(y_val, pred, labels=["insatisfaite"],
                                            average=None)[0], 3),
        "précision insat.": round(precision_score(y_val, pred, labels=["insatisfaite"],
                                                  average=None, zero_division=0)[0], 3),
        "f1_macro": round(f1_score(y_val, pred, average="macro"), 3),
        "coût (€)": cout_total(y_val, pred),
    })

tableau_poids = pd.DataFrame(lignes).set_index("poids insatisfaite")
tableau_poids

In [ ]:
meilleur_poids = int(tableau_poids["coût (€)"].idxmin())
print(f"Poids qui minimise le coût métier sur la validation : {meilleur_poids}")

Lecture honnête du tableau : avec une asymétrie aussi forte (80 € contre 4 €),
le coût continue de baisser quand le poids augmente… mais **en plateau**, et
pendant ce temps la précision et le `f1_macro` s'effondrent — le SAV rappelle
tout le monde. La validation qui te pousse **au bord de la grille** est un
signal : ce levier est **grossier** (il déforme tout l'apprentissage) et il
impose de **ré-entraîner à chaque réglage**. D'où le levier 2.

## [4] Levier 2 — le seuil de décision (zéro ré-entraînement)

En multi-classe, il n'y a pas « un » seuil comme en binaire. On définit une
**règle de priorité métier** par-dessus les probabilités :

> si `P(insatisfaite) ≥ s` → alerte SAV, quelle que soit la classe majoritaire ;
> sinon → argmax des classes restantes.

Le gros avantage opérationnel : `s` se règle **en production, sans ré-entraîner**
— quand Marc changera d'avis sur les 80 €, tu ajusteras un paramètre, pas un
modèle.

In [ ]:
probas_val = pipe_brut.predict_proba(X_val)
classes = list(pipe_brut.classes_)
i_insat = classes.index("insatisfaite")


def applique_seuil(probas: np.ndarray, seuil: float) -> np.ndarray:
    """Alerte 'insatisfaite' si sa probabilité dépasse le seuil, sinon argmax des autres."""
    p = probas.copy()
    alerte = p[:, i_insat] >= seuil
    p[:, i_insat] = -1.0  # on retire l'option 'insatisfaite' du choix par défaut
    par_defaut = np.array(classes)[p.argmax(axis=1)]
    return np.where(alerte, "insatisfaite", par_defaut)


lignes = []
for seuil in [0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    pred = applique_seuil(probas_val, seuil)
    lignes.append({
        "seuil": round(seuil, 2),
        "rappel insat.": round(recall_score(y_val, pred, labels=["insatisfaite"],
                                            average=None)[0], 3),
        "précision insat.": round(precision_score(y_val, pred, labels=["insatisfaite"],
                                                  average=None, zero_division=0)[0], 3),
        "coût (€)": cout_total(y_val, pred),
    })

tableau_seuils = pd.DataFrame(lignes).set_index("seuil")
tableau_seuils

In [ ]:
tableau_seuils["coût (€)"].plot(marker="o", title="Coût métier selon le seuil d'alerte")
plt.xlabel("seuil sur P(insatisfaite)")
plt.ylabel("coût total sur la validation (€)")
plt.tight_layout()

meilleur_seuil = float(tableau_seuils["coût (€)"].idxmin())
print(f"Seuil qui minimise le coût métier sur la validation : {meilleur_seuil}")

Cette fois la courbe a un **vrai minimum intérieur** : trop bas, le SAV se
noie (des milliers de rappels à 4 €) ; trop haut, on reperd les clientes à
80 €.

> 🧠 **Le lien avec la théorie** : en binaire, le seuil optimal vaut
> `coût_fausse_alerte / (coût_fausse_alerte + coût_erreur_grave)` —
> ici `4 / (4 + 80) ≈ 0,05`. La validation retrouve empiriquement un optimum
> tout proche (la 3ᵉ classe et le coût intermédiaire de 10 € le décalent un
> peu). Autrement dit : **la matrice de coûts DICTE le seuil**. Si Marc change
> ses 80 €, tu recalcules le seuil — sans ré-entraîner quoi que ce soit.

## [5] Verdict final — le test scellé, ouvert UNE fois

On fige les deux réglages choisis **sur la validation**, on ré-entraîne sur
tout le train, et on compare les 4 stratégies sur le test — c'est la seule
fois qu'on y touche.

In [ ]:
strategies = {}

pipe_final_brut = Pipeline([("preparation", fabrique_preparation()),
                            ("modele", LogisticRegression(max_iter=2000))])
pipe_final_brut.fit(X_train, y_train)
strategies["baseline (aucun réglage)"] = pipe_final_brut.predict(X_test)

pipe_balanced = Pipeline([("preparation", fabrique_preparation()),
                          ("modele", LogisticRegression(max_iter=2000,
                                                        class_weight="balanced"))])
pipe_balanced.fit(X_train, y_train)
strategies["class_weight='balanced'"] = pipe_balanced.predict(X_test)

pipe_poids = Pipeline([("preparation", fabrique_preparation()),
                       ("modele", LogisticRegression(max_iter=2000,
                                                     class_weight={"insatisfaite": meilleur_poids,
                                                                   "mitigée": 1,
                                                                   "satisfaite": 1}))])
pipe_poids.fit(X_train, y_train)
strategies[f"poids optimisé ({meilleur_poids})"] = pipe_poids.predict(X_test)

strategies[f"seuil optimisé ({meilleur_seuil})"] = applique_seuil(
    pipe_final_brut.predict_proba(X_test), meilleur_seuil
)

lignes = []
for nom, pred in strategies.items():
    lignes.append({
        "stratégie": nom,
        "rappel insat.": round(recall_score(y_test, pred, labels=["insatisfaite"],
                                            average=None)[0], 3),
        "f1_macro": round(f1_score(y_test, pred, average="macro"), 3),
        "coût (€)": cout_total(y_test, pred),
    })

pd.DataFrame(lignes).set_index("stratégie")

### 📝 Verdict — 3 lignes pour Marc Delval

Rédige-le toi-même, dans sa langue : combien d'euros économisés par rapport à
la baseline, combien de clientes insatisfaites récupérées, et le prix accepté
en échange (les rappels inutiles). Pas un mot de `f1_macro` — garde-le pour
ton notebook.

### 🤔 Question réflexive — la zone grise et l'humain

Un modèle peu sûr de lui ne devrait pas décider seul. Mesure la **zone
grise** : quelle part des avis a une probabilité max < 0,5 ? En production,
ces cas partent en **file de revue humaine** (human-in-the-loop) — c'est le
mécanisme de *fallback* du canvas §7.2, et un garde-fou attendu dès que la
décision touche des personnes.

In [ ]:
p_max = pipe_balanced.predict_proba(X_test).max(axis=1)
print(f"Avis en zone grise (proba max < 0,5) : {(p_max < 0.5).mean():.1%} du flux")
print("→ à router vers une revue humaine plutôt que de décider automatiquement.")

## 🔎 Ce que tu viens de revoir

- **Matrice de coûts métier** : traduire « cette erreur est plus grave » en
  euros, et en faire une métrique d'arbitrage (`confusion × coûts`).
- **Poids de classes** : ré-entraîner en pénalisant la classe critique —
  l'optimum dépend des coûts, pas d'une règle magique.
- **Seuil de décision multi-classe** : une règle de priorité sur
  `predict_proba`, réglable en prod **sans ré-entraîner**.
- **Le test reste scellé** : les réglages se cherchent sur la **validation**
  (cf. `fiche_validation_reglage.pdf`), le test ne sert qu'au verdict.
- **Zone grise → human-in-the-loop** : en dessous d'un niveau de confiance,
  c'est un humain qui décide.

## ⭐ Pour aller plus loin (optionnel)

- Refais le sweep de seuil avec le pipeline `balanced` : les deux leviers se
  **cumulent-ils** ou se cannibalisent-ils ?
- Trace la courbe précision-rappel de la seule classe `insatisfaite`
  (`precision_recall_curve` sur `P(insatisfaite)`, en un-contre-tous).
- `fiche_desequilibre_classes.pdf` mentionne SMOTE : pourquoi les poids de
  classes suffisent-ils ici ? (Indice : 2 400 exemples minoritaires, ce n'est
  pas « rare », c'est juste minoritaire.)